# Demonstrating parallel simcat database simulation on HPC with SLURM

### Imports

In [1]:
import toytree
import simcat
import subprocess
import os
import h5py
import numpy as np

### Define the species tree model and database params

In [2]:
tre = toytree.rtree.imbtree(4,6e6)
tre.draw(ts='p');

<svg class="toyplot-canvas-Canvas" xmlns:toyplot="http://www.sandia.gov/toyplot" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg" width="300.0px" height="300.0px" viewBox="0 0 300.0 300.0" preserveAspectRatio="xMidYMid meet" style="background-color:transparent;border-color:#292724;border-style:none;border-width:1.0;fill:rgb(16.1%,15.3%,14.1%);fill-opacity:1.0;font-family:Helvetica;font-size:12px;opacity:1.0;stroke:rgb(16.1%,15.3%,14.1%);stroke-opacity:1.0;stroke-width:1.0" id="t8ae8176e61ec4648988ba34877fecd41"> 0 1 2 3 4 5 6 r0 r1 r2 r3 0 2000000 4000000 6000000

In [3]:
num_rows_db = 500 # number of simulations in the database
db = simcat.Database("hpc_demo",
                '../dbs/',
                tre,
                nrows=num_rows_db,
                nsnps=200,
                Ne_min=10000, # how much should Ne vary on the branches?
                Ne_max=20000,
                admix_prop_min=0.1, # how much should the magnitude of admixture event vary?
                admix_prop_max=0.3,
                admix_edge_min=0.1, # how much should the timing of admixture event vary?
                admix_edge_max=0.9,
                exclude_sisters=True, # do we want to include introgression between sister taxa?
                node_slide_prop=0.25, # how much do we want internal nodes to shift around?
                existing_admix_edges=[],
                    ) # do we want to assume any existing edges?

500 labels to be stored in: ../dbs/hpc_demo.labels.h5


### Write out base scripts

#### simcat `simulate` script: inits the simulator based on the database name, specifies the number of simulations per job

In [4]:
simulate_script = """import simcat
simulator = simcat.Simulator("hpc_demo","/n/home09/pfmckenzie/pfmckenzie/projects/simcat_manuscript/dbs")  # inits the simulator
simulator.run(20,auto=True) # runs as many simulations as we specify, automatically detects available cores
"""
simulate_script_path = "../scripts/run_simcat_HPC_demo.py"

In [5]:
with open(simulate_script_path,'w') as f:
    f.write(simulate_script)

#### SLURM template: specify the length of number of cores per job, length of each job, memory, partition, etc. 

#### These parameters could be tuned by running jobs with few simulations and examining efficiency afterward with `seff {job_id}`

In [6]:
slurm_template = """#!/bin/bash
#SBATCH -c 4
#SBATCH -t 0-3:59:00
#SBATCH -p shared
#SBATCH --mem-per-cpu=2G
#SBATCH --job-name=hpc_{job_num}
#SBATCH -o {log_dir}/hpc_{job_num}.out
#SBATCH -e {log_dir}/hpc_{job_num}.err

# Activate your mambaforge environment
source /n/home09/pfmckenzie/mambaforge/bin/activate

# Deactivate file locking, we do this manually
export HDF5_USE_FILE_LOCKING=FALSE

# Navigate to script directory
cd {script_dir}

# Execute the Python script
python run_simcat_HPC_demo.py
"""

In [7]:
# specify the directory to save the slurm scripts in (one submit script per job, plus .err and .out files as they run)
slurm_script_dir = "../scripts/slurm_simulate_scripts"

# absolute paths to script and log directories
abs_script_dir = os.path.abspath(os.path.dirname(simulate_script_path))
abs_log_dir = os.path.abspath(slurm_script_dir)

### Run SLURM jobs in parallel to fill the database with simulations

In [8]:
# how many jobs needed?
sims_per_job = 20 # taken from `run_simcat_HPC_demo.py` script above
num_rows_db / sims_per_job

25.0

In [9]:
num_jobs = 25  # change this to number of jobs

# submit jobs in loop
for i in range(num_jobs):
    slurm_script_content = slurm_template.format(
        job_num=i,
        script_dir=abs_script_dir,
        log_dir=abs_log_dir
    )
    slurm_script_file = os.path.join(slurm_script_dir, f"hpc_{i}.sh")

    # write SLURM script to file
    with open(slurm_script_file, "w") as f:
        f.write(slurm_script_content)

    # submit to SLURM
    subprocess.run(["sbatch", slurm_script_file], check=True)

Submitted batch job 11744647
Submitted batch job 11744648
Submitted batch job 11744649
Submitted batch job 11744650
Submitted batch job 11744651
Submitted batch job 11744652
Submitted batch job 11744653
Submitted batch job 11744654
Submitted batch job 11744655
Submitted batch job 11744656
Submitted batch job 11744657
Submitted batch job 11744658
Submitted batch job 11744659
Submitted batch job 11744660
Submitted batch job 11744661
Submitted batch job 11744662
Submitted batch job 11744663
Submitted batch job 11744664
Submitted batch job 11744665
Submitted batch job 11744666
Submitted batch job 11744667
Submitted batch job 11744668
Submitted batch job 11744669
Submitted batch job 11744670
Submitted batch job 11744671


### Once the jobs finish running: Confirm that all jobs completed successfully 
#### -> Check the `finished_sims` dataset in the labels file. Finished jobs will have a value of 1.

In [10]:
# open labels file in read mode
with h5py.File('../dbs/hpc_demo.labels.h5', 'r') as labs_file:
    # access the finished_sims dataset
    finished_data = labs_file['finished_sims']

    # how many indices currently marked as 1 (= finished)?
    print(np.sum(finished_data[:] == 1))

500


#### You can also use `sacct` and `seff` to examine outcomes of individual jobs